In [1]:
import requests
import json
import base64
import time
import os
import pandas as pd
from IPython.display import display, Markdown

# ---------- 1. Model feature detection ----------
def get_model_features(model_name):
    """
    Determine if model has thinking/reasoning and vision.
    1. Try to load CSV and look up 'Input' column.
    2. If not found, use name heuristics.
    Returns (has_thinking, has_vision)
    """
    # Try to read combined CSV (if you have it)
    csv_path = r"E:\llm\Model\csv\all_models_combined.csv"
    input_type = None
    if os.path.exists(csv_path):
        try:
            df = pd.read_csv(csv_path)
            row = df[df['Name'] == model_name]
            if not row.empty:
                input_type = row.iloc[0].get('Input', '')
        except:
            pass

    # Vision detection from CSV if available
    if input_type is not None:
        has_vision = 'Image' in str(input_type)
    else:
        # Heuristic based on name
        low = model_name.lower()
        vision_patterns = ['vl', 'vision', 'llava', 'llama3.2-vision', 'llama4',
                           'gemma3', 'gemma4', 'ministral', 'mistral-small3.1',
                           'mistral-medium-3.5', 'medgemma', 'qwen2.5vl', 'qwen3-vl']
        has_vision = any(p in low for p in vision_patterns)

    # Thinking detection
    low = model_name.lower()
    thinking_patterns = ['thinking', 'reasoning', 'deepseek-r1', 'phi4-reasoning',
                         'phi4-mini-reasoning', 'nemotron-3.5-lightning',
                         'nemotron-3-nano', 'nemotron-3-super']
    has_thinking = any(p in low for p in thinking_patterns)
    # Some models may output reasoning without "thinking" in name (e.g., deepseek-r1)
    if 'deepseek-r1' in low:
        has_thinking = True

    return has_thinking, has_vision

# ---------- 2. Image helpers ----------
def image_to_base64_from_path(image_path):
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

def image_to_base64_from_url(image_url):
    resp = requests.get(image_url, timeout=30)
    resp.raise_for_status()
    return base64.b64encode(resp.content).decode("utf-8")

def image_to_base64(image_input):
    if image_input.startswith(("http://", "https://")):
        return image_to_base64_from_url(image_input)
    else:
        return image_to_base64_from_path(image_input)

# ---------- 3. Main chat function ----------
def chat_aware(pno,prompt, model="qwen3:4b", image_input=None):
    """
    Chat with automatic handling of thinking/reasoning and vision.
    - If model supports vision and no image_input given, ask user for path/URL.
    - Displays model features in the output.
    """
    has_thinking, has_vision = get_model_features(model)

    # If vision model and no image provided, ask interactively
    # if has_vision and image_input is None:
    #     print(f"🔍 Model '{model}' supports images.")
    #     img = input("Enter image path or URL (or press Enter to skip): ").strip()
    #     if img:
    #         image_input = img

    # Build message
    message = {"role": "user", "content": prompt}
    # if image_input:
    #     try:
    #         image_b64 = image_to_base64(image_input)
    #         message["images"] = [image_b64]
    #     except Exception as e:
    #         print(f"❌ Error loading image: {e}")
    #         return

    payload = {
        "model": model,
        "messages": [message],
        "stream": True,
    }

    resp = requests.post(
        "http://localhost:11434/api/chat",
        json=payload,
        stream=True,
        timeout=300,
    )
    resp.raise_for_status()

    # Timers
    start_time = time.time()
    first_thinking_time = None
    first_content_time = None
    thinking_text = ""
    content_text = ""

    # Create display handle
    handle = display(Markdown(""), display_id=True)

    # Build feature header
    features = []
    if has_thinking:
        features.append("🧠 **Thinking**")
    if has_vision:
        features.append("👁️ **Vision**")
    features_str = ", ".join(features) if features else "None"
    header = f"### Model: {model}\n**Features:** {features_str}\n\n"

    for line in resp.iter_lines(decode_unicode=True):
        if not line:
            continue
        data = json.loads(line)
        msg = data.get("message", {})

        # Thinking
        if "thinking" in msg and msg["thinking"]:
            thinking_text += msg["thinking"]
            if first_thinking_time is None:
                first_thinking_time = time.time()

        # Content
        if "content" in msg and msg["content"]:
            content_text += msg["content"]
            if first_content_time is None:
                first_content_time = time.time()

        # Build markdown
        md = header
        md += f"## 💎 Page {pno}"
        md += f"\n----\n"
        md +=f"## 💖💟 Prompt ♻️🚷🚯🚱🔞🚭☣️☢️"
        md += f"\n----\n"
        md += f"{prompt}"
        md += f"\n----\n"
        if thinking_text:
            md += f"## 🤔 Thinking\n\n{thinking_text}\n\n---\n\n"
        md += f"## 💬 Answer\n\n{content_text}"

        handle.update(Markdown(md))

        if data.get("done"):
            break

    total_time = time.time() - start_time

    # Final timings
    timings = ""
    if first_thinking_time:
        timings += f"**TTFT (thinking):** {first_thinking_time - start_time:.3f}s  \n"
    if first_content_time:
        timings += f"**TTFT (content):** {first_content_time - start_time:.3f}s  \n"
    timings += f"**Total time:** {total_time:.3f}s"

    handle.update(Markdown(md + "\n---\n" + timings))

    return thinking_text, content_text

In [2]:
FILE_NAME = "../../NNDesign.pdf"
FILE_NAME_PAPER = "../../2608.02980v1.pdf"


import pymupdf
import os

Folder1 = "border"

os.makedirs(Folder1, exist_ok=True)

doc = pymupdf.open(FILE_NAME_PAPER)

print(f"Total pages: {doc.page_count}")

def choose(pno):
    page = doc[pno]
    blocks = page.get_text("blocks")
    print(f"Page {pno+1}: {len(blocks)} block(s)")
    return blocks
    #for block in blocks:
        # Tuple: (x0, y0, x1, y1, text, block_no, block_type)
        #rect = pymupdf.Rect(block[0], block[1], block[2], block[3])
        #print(block[4])
        #return block
    #return blocks
        

Total pages: 21


In [3]:
def choose(pno,model="granite4:350m-h"):
    page = doc[pno]
    blocks = page.get_text("blocks")
    #print(f"Page {pno+1}: {len(blocks)} block(s)")
    #return blocks
    for block in blocks:
        # Tuple: (x0, y0, x1, y1, text, block_no, block_type)
        #rect = pymupdf.Rect(block[0], block[1], block[2], block[3])
        #print(block[4])
        #return block
        chat_aware(pno,block[4],model)
    pass


In [4]:
import pymupdf
import os

Folder1 = "border"

os.makedirs(Folder1, exist_ok=True)

doc = pymupdf.open(FILE_NAME_PAPER)

print(f"Total pages: {doc.page_count}")

for pno in range(doc.page_count):
 choose(pno,model="gemma3:270m-it-qat")

Total pages: 21


### Model: gemma3:270m-it-qat
**Features:** 👁️ **Vision**

## 💎 Page 0
----
## 💖💟 Prompt ♻️🚷🚯🚱🔞🚭☣️☢️
----
Qwen-3D: A Generalist 3D Vision-Language Model for Spatial Understanding

----
## 💬 Answer

The Qwen-3D is a versatile 3D vision-language model designed for spatial understanding. It leverages advanced architectures and techniques to effectively capture and process complex spatial relationships between objects and their environment. 

Here's a breakdown of key aspects and capabilities:

* **Structure and Representation:** Qwen-3D excels at understanding the hierarchical structure of objects and their interactions. It employs techniques like:
    * **Scene-Aware Training:**  It learns to identify and represent the relationships between different objects in a scene, including objects with similar properties and interactions.
    * **Attention Mechanisms:** It employs attention mechanisms to focus on the most relevant parts of the scene, enabling it to understand the underlying spatial context.
    * **Knowledge Graphs:**  It can leverage knowledge graphs to provide context-aware representations of objects and their relationships, enabling more accurate reasoning and decision-making.
    * **Probabilistic Reasoning:**  It can be trained to predict the likely locations of objects in a scene, improving its ability to handle complex and uncertain situations.
* **Reasoning and Inference:** Qwen-3D exhibits strong reasoning abilities, enabling it to understand complex spatial relationships and infer new information about objects and their environment.
* **Adaptability and Flexibility:**  It can adapt to new and evolving environments by incorporating existing knowledge and modifying its internal representations.
* **Robustness:**  Qwen-3D is designed to be robust to noise and other variations in input data, ensuring its ability to perform accurately across a wide range of spatial scenarios.
* **Applications:**  It is well-suited for a variety of applications, including:
    * **Robotics:** Understanding and interacting with robots and automated systems.
    * **Autonomous Driving:**  Assisting in navigation and decision-making.
    * **Environmental Sensing:**  Analyzing and interpreting sensor data to understand environmental conditions.
    * **Medical Imaging:**  Analyzing medical images to aid in diagnosis and treatment planning.
    * **Resource Management:**  Optimizing resource allocation and planning.
    * **Virtual Reality:**  Creating immersive and interactive virtual environments.
* **Evaluation Metrics:**  Qwen-3D is evaluated using various metrics, including accuracy, efficiency, and robustness, to assess its performance across different scenarios.

In summary, Qwen-3D is a powerful and versatile 3D vision-language model with a strong foundation in spatial reasoning, adaptation capabilities, and robustness. Its ability to understand complex spatial relationships and make informed decisions makes it a valuable tool for a wide range of applications.
---
**TTFT (content):** 0.000s  
**Total time:** 59.373s

### Model: gemma3:270m-it-qat
**Features:** 👁️ **Vision**

## 💎 Page 0
----
## 💖💟 Prompt ♻️🚷🚯🚱🔞🚭☣️☢️
----
Lucy Lin†, Ayush Jain†, Yifan Liu, Katerina Fragkiadaki

----
## 💬 Answer

Okay, I am ready to help with any requests. Please let me know what you need!
---
**TTFT (content):** 0.000s  
**Total time:** 2.061s

### Model: gemma3:270m-it-qat
**Features:** 👁️ **Vision**

## 💎 Page 0
----
## 💖💟 Prompt ♻️🚷🚯🚱🔞🚭☣️☢️
----
Carnegie Mellon University

----
## 💬 Answer

Okay, I'm ready. Let me know what you'd like to do.
---
**TTFT (content):** 0.001s  
**Total time:** 1.927s

### Model: gemma3:270m-it-qat
**Features:** 👁️ **Vision**

## 💎 Page 0
----
## 💖💟 Prompt ♻️🚷🚯🚱🔞🚭☣️☢️
----
{lucylin,ayushj2,yifanliu,kfragki2}@andrew.cmu.edu

----
## 💬 Answer


---
**Total time:** 0.003s

...

### Model: gemma3:270m-it-qat
**Features:** 👁️ **Vision**

## 💎 Page 1
----
## 💖💟 Prompt ♻️🚷🚯🚱🔞🚭☣️☢️
----
Figure 1. Qwen-3D performs attention directly in 3D world space rather than over independent image frames. Given multi-view
RGB observations, depth, and camera poses, Qwen-3D maps visual tokens into a shared 3D coordinate system and applies geometry-aware
attention through 3D Rotary Positional Embeddings. The model jointly supports language reasoning, 2D grounding, and 3D grounding
within a unified architecture, achieving state-of-the-art performance across a broad range of vision–language and 3D understanding bench-
marks.

----
## 💬 Answer

You're right, Qwen-3D is demonstrating remarkable success in solving the problem of attention-based reasoning in 3D. While the work on this topic is ongoing and evolving, the foundation is solid.

Let's break down the key aspects of Qwen-3D's architecture and the rationale behind its attention-based reasoning capabilities.

**Key Features and Contributions:**

* **Attention-Based Reasoning:** Qwen-3D utilizes attention mechanisms to effectively understand and reason about multi-view RGB data. This is a significant advancement compared to traditional methods like CNNs, which often rely on local feature extraction and model-level abstraction.
* **Multi-View Vision:** The model effectively handles multi-view RGB observations, which is crucial for understanding the complex relationships between objects in the 3D space.
* **Depth and Camera Pose:** The attention mechanism allows Qwen-3D to dynamically map visual tokens to a shared 3D coordinate system, enabling efficient reasoning about depth and camera poses.
* **Unified Architecture:** The model is built upon a unified architecture, which allows for seamless integration of different vision-language and 3D understanding tasks. This unified approach enhances the model's ability to generalize and perform tasks across diverse environments.
* **State-of-the-Art Performance:** Qwen-3D demonstrates impressive performance in various vision-language and 3D understanding benchmarks, including language reasoning, 2D grounding, and 3D grounding. This indicates a strong foundation in the field of attention-based reasoning.

**Why Attention-Based Reasoning is Important:**

* **Improved Understanding:** By focusing on the salient features in the 3D space, attention mechanisms allow models to better understand the relationships between objects and understand the context of their actions.
* **Enhanced Reasoning:** This improved understanding leads to more accurate and nuanced reasoning compared to traditional methods.
* **Better Performance:** Attention-based reasoning often leads to better performance in a wide range of vision-language and 3D tasks.

**Potential Challenges and Future Directions:**

Despite its impressive achievements, Qwen-3D faces several challenges:

* **Computational Complexity:** Training and deploying attention-based reasoning models can be computationally expensive, especially for large datasets.
* **Model Size:** The model size can be relatively large, which can limit its accessibility for specialized tasks or specific hardware.
* **Training Data:** The quality and quantity of training data are crucial for training effective attention mechanisms.
* **Interpretability:** Understanding the reasoning process of attention-based reasoning is an ongoing area of research, which can add complexity to the model.

**In Conclusion, Qwen-3D's ability to effectively use attention to reason about multi-view RGB data and achieve state-of-the-art performance in vision-language and 3D understanding makes it a significant advancement in the field.**
---
**TTFT (content):** 0.001s  
**Total time:** 67.127s

### Model: gemma3:270m-it-qat
**Features:** 👁️ **Vision**

## 💎 Page 1
----
## 💖💟 Prompt ♻️🚷🚯🚱🔞🚭☣️☢️
----
3D detectors.
We argue that this gap stems from a fundamental chal-
lenge in adapting language-centric architectures to 3D per-
ception. While large language models excel at reasoning
over discrete tokens, dense 3D grounding requires predict-
ing spatially precise outputs in a continuous world coor-
dinate system. Unlike images, which provide a canonical
pixel coordinate frame, 3D scenes admit no universal refer-
ence frame: the same object may appear at entirely different
coordinates across scans and environments. As a result, au-
toregressively decoding 3D boxes, coordinates, or masks as
language tokens is an unnatural interface for 3D perception.

----
## 💬 Answer

Yes, that's a valid point. It's a significant gap in our understanding of 3D perception. While large language models (LLMs) are good at processing text, they struggle with understanding and generating coherent and meaningful representations of 3D concepts and objects. The question of how to bridge this gap is a crucial and ongoing area of research.

The argument that the gap arises from a fundamental chal-
thange in adapting language-centric architectures to 3D per-
ception is a valid and important point to consider. However, it's important to remember that large language models are not perfect at understanding and generating human-like representations of 3D. They can sometimes generate outputs that are not as realistic or nuanced as those generated by humans.

One possible way to address this gap is to develop more sophisticated architectures that can effectively learn and understand 3D concepts. This could involve incorporating techniques like:

* **Generative models:** These models can generate novel outputs from existing knowledge, which can be used to bridge the gap between LLMs and 3D perception.
* **Hybrid models:** These models combine the strengths of both LLMs and 3D models. For example, a hybrid model could be trained to incorporate information from both LLMs and 3D representations.
* **Reinforcement learning:** These models can learn to interact with and interact with 3D environments, allowing them to learn to predict the actions of other agents.

It's important to note that these approaches are still in their early stages of development and that there is still a need for more research to fully understand the nuances of 3D perception and to develop effective strategies for bridging the gap between LLMs and 3D.

In summary, the question of how to bridge the gap between LLMs and 3D perception is a significant and complex issue that requires ongoing research and development.
---
**TTFT (content):** 0.000s  
**Total time:** 43.493s

### Model: gemma3:270m-it-qat
**Features:** 👁️ **Vision**

## 💎 Page 1
----
## 💖💟 Prompt ♻️🚷🚯🚱🔞🚭☣️☢️
----
Existing 3D LMMs typically address this problem either
by representing grounding outputs through text generation
or by attaching lightweight grounding modules that com-
municate with the backbone through a small set of query
vectors (Figure 2). While these approaches preserve the
reasoning capabilities of the underlying language model,
they create a severe information bottleneck between high-
capacity visual representations and the dense geometric pre-
dictions required for grounding. Consequently, current 3D
LMMs improve high-level spatial reasoning but remain sig-
nificantly weaker than specialist systems on core 3D per-
ception tasks. This observation raises an important open
question: how should a large multimodal model interface
with a 3D grounding system?

----
## 💬 Answer

The provided answer correctly identifies the problem: a severe information bottleneck between high-capacity visual representations and dense geometric pre-dictions for grounding.

The answer also correctly states that current 3D LMMs improve high-level spatial reasoning, but become weaker than specialist systems on core 3D per-ception tasks. This observation is a crucial point that emphasizes the need for a more robust and efficient approach to address the limitations of current 3D LMMs.
---
**TTFT (content):** 0.000s  
**Total time:** 10.439s

### Model: gemma3:270m-it-qat
**Features:** 👁️ **Vision**

## 💎 Page 1
----
## 💖💟 Prompt ♻️🚷🚯🚱🔞🚭☣️☢️
----
We introduce Qwen-3D, a geometry-aware 3D LMM
that extends the Qwen family of models [5] with explicit

----
## 💬 Answer

```python
import math

def calculate_qwen_3d(model_path, output_file="qwen_3d.txt"):
    try:
        with open(model_path, 'r') as f:
            model = f.read()
        print(f"Model: {model}")
        # Calculate the qwen-3d model
        qwen_3d_model = model.split('')[0].split(' ')[0]
        print(f"Qwen-3D Model: {qwen_3d_model}")
    except Exception as e:
        print(f"Error calculating qwen-3d model: {e}")
```

**Explanation:**

1.  **Import Libraries:** `math` for calculating the `math.sqrt` function and `math.pi` for the standard unit of measure.
2.  **`calculate_qwen_3d()` Function:** This function takes the `model_path` and an optional `output_file` as arguments.
3.  **Error Handling:** The code includes a `try...except` block to handle potential errors during the model loading process.
4.  **Model Loading:**
    *   The `model_path` is read from a file.
    *   The `model` variable is assigned to the `qwen_3d_model` variable.
    *   The `print()` statement is used to print the model to the console.
    *   The `print()` statement is used to calculate the qwen-3d model.
    *   The `try...except` block is used to catch potential exceptions during the model loading process.
5.  **Calculating the Qwen-3D Model:**
    *   `model.split('')[0].split(' ')[0]`: This splits the `model` string into a list of strings, extracting the first element of each string. This is the Qwen-3D model.
    *   `model.split(' ')[0].split(' ')[1]`: This splits the list of strings into a list of strings, extracting the second element of each string. This is the Qwen-3D model.
    *   `model.

KeyboardInterrupt: 